# Multi-Tissue HCE v4.5 — Label Unification + Filtering

| | v4 | **v4.5 (this notebook)** |
|---|---|---|
| Synonym map | 7 mappings | **27 mappings** (lung abbreviations, `, human` suffixes, monocyte/DC variants, endothelial variants, hierarchy mismatch fix) |
| Bad label filtering | None | **3 labels dropped** (malignant cell from liver; stromal cell of pancreas, alveolar macrophage from lymph_node) |
| Everything else | Same | Same |

## 1. Imports

In [1]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

import scanpy as sc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import scipy.sparse as sp
import h5py

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
from cell2sentence.hce_trainer import build_reachability_matrix_from_ontology
from cell2sentence.hierarchy_utils import deduplicate_hierarchy, find_collisions

warnings.filterwarnings('ignore')
print('=' * 60)
print('  IMPORTS OK')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}')
print('=' * 60)

/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/c2s-justin/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  IMPORTS OK
  PyTorch : 2.10.0+cu128
  CUDA    : True | devices: 1


## 2. Configuration

In [2]:
# ── Output ───────────────────────────────────────────────────────────────────
OUT_DIR         = 'multi_tissue_v4_5_results'
BEST_MODEL_PATH = os.path.join(OUT_DIR, 'best_model.pt')
C2S_MODEL_NAME  = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'

# ── Training organs ───────────────────────────────────────────────────────────
ORGAN_CONFIGS = [
    {
        'name': 'lung',
        'path': 'lung.h5ad',
        'label_col': 'ann_finest_level',
        'gene_col': 'feature_name',
        'hierarchy_cols': ['ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5'],
        'coarse_col': None,
        'id': 0,
    },
    {
        'name': 'brain_glia',
        'path': 'brain_new.h5ad',
        'label_col': 'cell_type',
        'gene_col': 'feature_name',
        'hierarchy_cols': [],
        'coarse_col': 'supercluster_term',
        'id': 1,
    },
    {
        'name': 'brain_neurons',
        'path': 'brain_neurons_processed.h5ad',
        'label_col': 'label',
        'gene_col': 'feature_name',
        'hierarchy_cols': [],
        'coarse_col': None,
        'id': 2,
    },
    {
        'name': 'liver',
        'path': 'census_data/liver.h5ad',
        'label_col': 'cell_type',
        'gene_col': 'feature_name',
        'hierarchy_cols': [],
        'coarse_col': None,
        'id': 3,
    },
    {
        'name': 'lymph_node',
        'path': 'census_data/lymph_node.h5ad',
        'label_col': 'cell_type',
        'gene_col': 'feature_name',
        'hierarchy_cols': [],
        'coarse_col': None,
        'id': 4,
    },
    {
        'name': 'bone_marrow',
        'path': 'census_data/bone_marrow.h5ad',
        'label_col': 'cell_type',
        'gene_col': 'feature_name',
        'hierarchy_cols': [],
        'coarse_col': None,
        'id': 5,
    },
]

# ── Lab validation datasets (zero-shot, never used in training) ───────────────
LAB_CONFIGS = [
    {'name': 'All_cells (glioma)',  'path': 'All_cells.h5ad',                    'label_col': 'predicted.high_hierarchy'},
    {'name': 'Brain_normal',        'path': 'lab-data/Brain_normal.h5ad',        'label_col': 'cell_type'},
    {'name': 'Liver_normal',        'path': 'lab-data/Liver_normal.h5ad',        'label_col': 'cell_type'},
    {'name': 'Lymph_node_normal',   'path': 'lab-data/Lymph_node_normal.h5ad',   'label_col': 'cell_type'},
    {'name': 'lymphoid',            'path': 'lab-data/lymphoid.h5ad',            'label_col': 'cell_type'},
    {'name': 'myeloid',             'path': 'lab-data/myeloid.h5ad',             'label_col': 'cell_type'},
]

# ── Hyperparameters ───────────────────────────────────────────────────────────
TOP_K_GENES        = 200
MAX_CELLS_PER_TYPE = 300
MIN_CELLS_PER_TYPE = 100
TEST_FRAC          = 0.15
VAL_FRAC           = 0.10
BATCH_SIZE         = 16
N_EPOCHS           = 10
LEARNING_RATE      = 1e-4
WEIGHT_DECAY       = 1e-2
WARMUP_STEPS       = 200
MAX_SEQ_LEN        = 512
MAX_WEIGHT         = 10.0
SEED               = 42

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'  Device          : {device}')
print(f'  Output dir      : {OUT_DIR}')
print(f'  Training organs : {[c["name"] for c in ORGAN_CONFIGS]}')
print(f'  Lab datasets    : {[c["name"] for c in LAB_CONFIGS]}')
print(f'  Epochs          : {N_EPOCHS}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Max cells/type  : {MAX_CELLS_PER_TYPE}')
print(f'  Min cells/type  : {MIN_CELLS_PER_TYPE}')
print('[OK] Config ready')

  Device          : cuda
  Output dir      : multi_tissue_v4_5_results
  Training organs : ['lung', 'brain_glia', 'brain_neurons', 'liver', 'lymph_node', 'bone_marrow']
  Lab datasets    : ['All_cells (glioma)', 'Brain_normal', 'Liver_normal', 'Lymph_node_normal', 'lymphoid', 'myeloid']
  Epochs          : 10
  Batch size      : 16
  Max cells/type  : 300
  Min cells/type  : 100
[OK] Config ready


## 3. Utility Functions

In [3]:
def is_valid(value):
    if pd.isna(value): return False
    return str(value).strip().lower() not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')


def get_gene_symbols(adata):
    """Return gene symbols from feature_name col if present, else var_names."""
    if 'feature_name' in adata.var.columns:
        return np.array(adata.var['feature_name'].astype(str))
    return np.array(adata.var_names.astype(str))


def load_organ(cfg, min_cells=MIN_CELLS_PER_TYPE, max_cells=MAX_CELLS_PER_TYPE):
    """
    Load one organ's h5ad, cap at max_cells per type, drop types below min_cells.
    Returns: (adata_backed, obs_df, valid_indices_in_full_file, gene_symbols)
    """
    name      = cfg['name']
    label_col = cfg['label_col']
    t0 = time.time()

    adata = sc.read_h5ad(cfg['path'], backed='r')
    gene_symbols = get_gene_symbols(adata)

    valid_mask = adata.obs[label_col].apply(is_valid)
    obs        = adata.obs[valid_mask].copy()
    valid_idx  = np.where(valid_mask.values)[0]

    # Cap at max_cells per type (sample in obs-space, not file-space)
    labels  = obs[label_col].astype(str).values
    sampled = []
    for lbl in np.unique(labels):
        pos = np.where(labels == lbl)[0]
        if len(pos) > max_cells:
            pos = np.random.choice(pos, max_cells, replace=False)
        sampled.extend(pos.tolist())
    sampled   = np.sort(np.array(sampled, dtype=np.int64))
    obs       = obs.iloc[sampled].copy()
    valid_idx = valid_idx[sampled]

    # Drop types below min_cells
    counts  = obs[label_col].value_counts()
    keep    = counts[counts >= min_cells].index
    dropped = counts[counts < min_cells]
    if len(dropped):
        print(f'  [{name}] Dropping {len(dropped)} type(s) < {min_cells} cells:')
        for ct, n in dropped.items():
            print(f'    - {ct}: {n}')
    mask      = obs[label_col].isin(keep)
    obs       = obs[mask].copy()
    valid_idx = valid_idx[mask.values]

    n_types = obs[label_col].nunique()
    print(f'  [{name}] {len(obs):,} cells | {n_types} types | {time.time()-t0:.1f}s')
    return adata, obs, valid_idx, gene_symbols


def cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k=200, desc='cells'):
    """
    Stream rows from a backed CSR h5ad X matrix and convert to gene symbol text.
    Loads indptr once into memory (much faster than per-row h5py access).
    Returns list of text strings, one per row index.
    """
    texts = []
    with h5py.File(h5_path, 'r') as f:
        X          = f['X']
        indptr     = X['indptr'][:]   # load full indptr once — avoids N h5py calls
        indices_ds = X['indices']
        data_ds    = X['data']
        for row_idx in tqdm(row_indices, desc=f'  {desc}', leave=False):
            start, end = int(indptr[row_idx]), int(indptr[row_idx + 1])
            if start == end:
                texts.append('')
                continue
            vals = data_ds[start:end]
            cols = indices_ds[start:end]
            if len(vals) <= top_k:
                order = np.argsort(vals)[::-1]
            else:
                order = np.argpartition(vals, -top_k)[-top_k:]
                order = order[np.argsort(vals[order])[::-1]]
            texts.append(' '.join(
                str(gene_symbols[cols[j]]) for j in order if vals[j] > 0
            ))
    return texts


def cell_to_text_dense(X_row, gene_symbols, top_k=200):
    """Convert one dense/sparse row to gene symbol text."""
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz  = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    vals = row[nz]
    if len(nz) > top_k:
        idx = np.argpartition(vals, -top_k)[-top_k:]
        nz  = nz[idx[np.argsort(vals[idx])[::-1]]]
    else:
        nz = nz[np.argsort(vals)[::-1]]
    return ' '.join(gene_symbols[i] for i in nz)


print('[OK] Utility functions defined')

[OK] Utility functions defined


## 4. Load All Training Organs

In [4]:
print('=' * 60)
print('  Loading all training organs')
print('=' * 60)

loaded_organs = {}   # name -> {adata, obs, valid_idx, gene_symbols}

for cfg in ORGAN_CONFIGS:
    print(f'\n  --- {cfg["name"]} ---')
    adata, obs, valid_idx, gene_syms = load_organ(cfg)
    loaded_organs[cfg['name']] = {
        'cfg':         cfg,
        'adata':       adata,
        'obs':         obs,
        'valid_idx':   valid_idx,
        'gene_symbols': gene_syms,
    }

print('\n' + '=' * 60)
print('  SUMMARY')
print('=' * 60)
total = 0
for name, d in loaded_organs.items():
    n = len(d['obs'])
    t = d['obs'][d['cfg']['label_col']].nunique()
    print(f'  {name:<20} {n:>7,} cells  {t:>4} types')
    total += n
print(f'  {"TOTAL":<20} {total:>7,} cells')
print('[OK] All organs loaded')

  Loading all training organs

  --- lung ---
  [lung] Dropping 3 type(s) < 100 cells:
    - Hematopoietic stem cells: 61
    - Lymphatic EC proliferating: 28
    - Unknown: 0
  [lung] 17,700 cells | 59 types | 29.4s

  --- brain_glia ---
  [brain_glia] 3,900 cells | 13 types | 2.7s

  --- brain_neurons ---
  [brain_neurons] 6,000 cells | 20 types | 0.5s

  --- liver ---
  [liver] Dropping 793 type(s) < 100 cells:
    - stem cell: 83
    - CD8-alpha-alpha-positive, alpha-beta intraepithelial T cell: 81
    - effector memory CD4-positive, alpha-beta T cell, terminally differentiated: 79
    - skeletal muscle satellite cell: 74
    - pre-conventional dendritic cell: 59
    - enteroendocrine cell: 59
    - myeloid dendritic cell: 56
    - CD4-positive, CD25-positive, alpha-beta regulatory T cell: 56
    - double negative thymocyte: 49
    - hematopoietic precursor cell: 41
    - double-positive, alpha-beta thymocyte: 41
    - group 2 innate lymphoid cell: 40
    - glial cell: 38
    - pro

## 5. Label Normalization

In [5]:
print('=' * 60)
print('  Label normalization — unify shared types across organs')
print('=' * 60)

# ── Lung: fix pericyte naming ─────────────────────────────────────────────────
LUNG_LABEL_MAP = {'Pericytes': 'pericyte'}
lung_obs = loaded_organs['lung']['obs']
lung_col = loaded_organs['lung']['cfg']['label_col']
for old, new in LUNG_LABEL_MAP.items():
    n = (lung_obs[lung_col] == old).sum()
    if n:
        loaded_organs['lung']['obs'][lung_col] = lung_obs[lung_col].replace(old, new)
        print(f'  [lung] "{old}" -> "{new}"  ({n} cells)')

# ── Census organs: drop disease/non-normal cells ──────────────────────────────
for organ_name in ['liver', 'lymph_node', 'bone_marrow']:
    d   = loaded_organs[organ_name]
    obs = d['obs']
    if 'disease' in obs.columns:
        non_normal = (obs['disease'] != 'normal').sum()
        if non_normal:
            print(f'  [{organ_name}] Removing {non_normal} non-normal cells')
            mask = obs['disease'] == 'normal'
            d['obs']       = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]

print('\n[OK] Label normalization done')

  Label normalization — unify shared types across organs
  [lung] "Pericytes" -> "pericyte"  (300 cells)

[OK] Label normalization done


## 5.5  Label Audit & Synonym Unification  ← NEW in v4

**Problem (diagnosed in v3):** Several Census organs use different CL term strings for the same
biological population. This causes the model to split confidence across synonyms, reducing recall
for all of them.

**Fix:** `LABEL_SYNONYM_MAP` maps every variant form to one canonical string before training.
The audit cell below shows the raw label landscape so you can extend the map if needed.

In [6]:
# ── Label audit: show all unique labels per organ and flag cross-organ overlaps ─
print('=' * 60)
print('  LABEL AUDIT')
print('=' * 60)

organ_label_sets = {}
for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    col  = cfg['label_col']
    labels = set(loaded_organs[name]['obs'][col].astype(str).unique())
    organ_label_sets[name] = labels
    print(f'\n  [{name}]  {len(labels)} unique labels:')
    for lbl in sorted(labels):
        print(f'    {lbl}')

# Cross-organ overlap: labels that appear in 2+ organs
print('\n' + '=' * 60)
print('  CROSS-ORGAN LABEL OVERLAPS')
print('=' * 60)
all_labels = {}
for name, labels in organ_label_sets.items():
    for lbl in labels:
        all_labels.setdefault(lbl, []).append(name)
shared = {lbl: orgs for lbl, orgs in all_labels.items() if len(orgs) > 1}
for lbl, orgs in sorted(shared.items()):
    print(f'  {lbl:<60} in: {orgs}')

print(f'\n  {len(shared)} labels shared across 2+ organs')
print('[OK] Audit complete — review output before editing LABEL_SYNONYM_MAP below')

  LABEL AUDIT

  [lung]  59 unique labels:
    AT0
    AT1
    AT2
    AT2 proliferating
    Adventitial fibroblasts
    Alveolar Mph CCL3+
    Alveolar Mph MT-positive
    Alveolar Mph proliferating
    Alveolar fibroblasts
    Alveolar macrophages
    B cells
    Basal resting
    CD4 T cells
    CD8 T cells
    Classical monocytes
    Club (nasal)
    Club (non-nasal)
    DC1
    DC2
    Deuterosomal
    EC aerocyte capillary
    EC arterial
    EC general capillary
    EC venous pulmonary
    EC venous systemic
    Goblet (bronchial)
    Goblet (nasal)
    Goblet (subsegmental)
    Hillock-like
    Interstitial Mph perivascular
    Ionocyte
    Lymphatic EC differentiating
    Lymphatic EC mature
    Mast cells
    Mesothelium
    Migratory DCs
    Monocyte-derived Mph
    Multiciliated (nasal)
    Multiciliated (non-nasal)
    Myofibroblasts
    NK cells
    Neuroendocrine
    Non-classical monocytes
    Peribronchial fibroblasts
    Plasma cells
    Plasmacytoid DCs
    SM activa

In [7]:
# ── Filter out known bad/contamination labels before synonym unification ──────
# These are not genuine cell types and should not appear as training classes.
print('=' * 60)
print('  Filtering bad / contamination labels')
print('=' * 60)

# Each entry: (organ_name, label_col, set_of_labels_to_drop, reason)
BAD_LABEL_FILTERS = [
    (
        'liver',
        'cell_type',
        {'malignant cell'},
        'not a normal cell type — should never appear as a training class',
    ),
    (
        'lymph_node',
        'cell_type',
        {'stromal cell of pancreas'},
        'pancreatic stroma in a lymph node dataset — likely annotation error',
    ),
    (
        'lymph_node',
        'cell_type',
        {'alveolar macrophage'},
        'lung-specific cell type appearing in lymph node dataset — likely contamination',
    ),
]

for organ_name, col, bad_labels, reason in BAD_LABEL_FILTERS:
    d   = loaded_organs[organ_name]
    obs = d['obs']
    for lbl in bad_labels:
        n = (obs[col] == lbl).sum()
        if n > 0:
            mask = obs[col] != lbl
            d['obs']       = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            obs = d['obs']  # refresh
            print(f'  [{organ_name}] Dropped "{lbl}" ({n} cells) — {reason}')
        else:
            print(f'  [{organ_name}] "{lbl}" not found (already absent)')

print('\n[OK] Bad label filtering done')

  Filtering bad / contamination labels
  [liver] Dropped "malignant cell" (300 cells) — not a normal cell type — should never appear as a training class
  [lymph_node] Dropped "stromal cell of pancreas" (300 cells) — pancreatic stroma in a lymph node dataset — likely annotation error
  [lymph_node] Dropped "alveolar macrophage" (114 cells) — lung-specific cell type appearing in lymph node dataset — likely contamination

[OK] Bad label filtering done


In [8]:
# ── LABEL_SYNONYM_MAP ─────────────────────────────────────────────────────────
# Maps every variant label to one canonical string.
# Rule: map the less-common / longer / less-precise form TO the canonical form.
# ─────────────────────────────────────────────────────────────────────────────

LABEL_SYNONYM_MAP = {

    # ── Lung plural/abbreviated → CL canonical ───────────────────────────────
    # Lung uses shorthand labels that don't match Census CL terms.
    # Without unification these become separate leaf nodes for the same biology.
    'B cells':                              'B cell',
    'NK cells':                             'natural killer cell',
    'Alveolar macrophages':                 'alveolar macrophage',
    'Mast cells':                           'mast cell',
    'Plasma cells':                         'plasma cell',
    'Classical monocytes':                  'classical monocyte',
    'Non-classical monocytes':              'non-classical monocyte',
    'Plasmacytoid DCs':                     'plasmacytoid dendritic cell',
    # 'Smooth muscle' in lung = airway smooth muscle; 'smooth muscle cell' in
    # lymph_node is the CL term for the same broad population.
    'Smooth muscle':                        'smooth muscle cell',
    # DC1/DC2/Migratory DCs are lung-specific subtypes with no direct Census
    # equivalent — intentionally NOT mapped.

    # ── CD4+ T cell variants ─────────────────────────────────────────────────
    'CD4 T cells':                          'CD4-positive, alpha-beta T cell',
    'CD4-positive helper T cell':           'CD4-positive, alpha-beta T cell',

    # ── CD8+ T cell variants ─────────────────────────────────────────────────
    'CD8 T cells':                          'CD8-positive, alpha-beta T cell',

    # ── Regulatory T cells ────────────────────────────────────────────────────
    # CL:0000815 'regulatory T cell' and the phenotype-annotated form describe
    # the same Treg population. Collapse to the shorter CL label.
    'CD4-positive, CD25-positive, alpha-beta regulatory T cell':
                                            'regulatory T cell',

    # ── CD8+ memory T cell variants ───────────────────────────────────────────
    # CD45RO is a surface marker of memory T cells, not a distinct subtype.
    'CD8-positive, alpha-beta memory T cell, CD45RO-positive':
                                            'CD8-positive, alpha-beta memory T cell',

    # ── Mature / staging descriptors ─────────────────────────────────────────
    'mature alpha-beta T cell':             'alpha-beta T cell',
    'mature NK T cell':                     'natural killer T cell',
    # 'mature B cell' is essentially a post-selection B cell — same as 'B cell'
    # for our classification granularity.
    'mature B cell':                        'B cell',

    # ── Effector memory CD4 terminally differentiated ─────────────────────────
    # 'terminally differentiated' qualifier is used inconsistently across datasets.
    'effector memory CD4-positive, alpha-beta T cell, terminally differentiated':
                                            'effector memory CD4-positive, alpha-beta T cell',

    # ── ', human' suffix variants ─────────────────────────────────────────────
    # Some Census organs append ', human' to CL terms; others do not.
    'dendritic cell, human':                'dendritic cell',
    'group 3 innate lymphoid cell, human':  'group 3 innate lymphoid cell',
    'plasmacytoid dendritic cell, human':   'plasmacytoid dendritic cell',

    # ── Monocyte surface-marker labels → functional CL names ─────────────────
    # CD14+ monocytes = classical monocytes; CD14+CD16+ = intermediate monocytes.
    'CD14-positive monocyte':               'classical monocyte',
    'CD14-positive, CD16-positive monocyte':'intermediate monocyte',

    # ── DC terminology variants ───────────────────────────────────────────────
    # Myeloid DCs and liver DCs are conventional DCs by lineage.
    'myeloid dendritic cell':               'conventional dendritic cell',
    'liver dendritic cell':                 'conventional dendritic cell',

    # ── Endothelial variants ──────────────────────────────────────────────────
    'vein endothelial cell':                        'endothelial cell of vein',
    'endothelial cell of pericentral hepatic sinusoid':
                                                    'endothelial cell of hepatic sinusoid',
    'endothelial cell of periportal hepatic sinusoid':
                                                    'endothelial cell of hepatic sinusoid',

    # ── Hepatic variants ──────────────────────────────────────────────────────
    'intrahepatic cholangiocyte':           'cholangiocyte',

    # ── Hierarchy name mismatch (data label ≠ CROSS_ORGAN_HIERARCHY key) ─────
    # Without this mapping, 'granulocyte monocyte progenitor cell' becomes a
    # disconnected root node — it never gets wired to 'hematopoietic precursor cell'.
    'granulocyte monocyte progenitor cell': 'granulocyte monocyte progenitor',

    # ── Functional states collapsed to cell type ──────────────────────────────
    # 'cycling' is a proliferative state, not a distinct cell identity.
    'cycling plasma cell':                  'plasma cell',
    # 'inflammatory' is an activation state; macrophage subtype for our purposes.
    'inflammatory macrophage':              'macrophage',

}

print(f'[OK] LABEL_SYNONYM_MAP defined — {len(LABEL_SYNONYM_MAP)} mappings')
print()
for src, tgt in LABEL_SYNONYM_MAP.items():
    print(f'  "{src}"')
    print(f'    -> "{tgt}"')

[OK] LABEL_SYNONYM_MAP defined — 32 mappings

  "B cells"
    -> "B cell"
  "NK cells"
    -> "natural killer cell"
  "Alveolar macrophages"
    -> "alveolar macrophage"
  "Mast cells"
    -> "mast cell"
  "Plasma cells"
    -> "plasma cell"
  "Classical monocytes"
    -> "classical monocyte"
  "Non-classical monocytes"
    -> "non-classical monocyte"
  "Plasmacytoid DCs"
    -> "plasmacytoid dendritic cell"
  "Smooth muscle"
    -> "smooth muscle cell"
  "CD4 T cells"
    -> "CD4-positive, alpha-beta T cell"
  "CD4-positive helper T cell"
    -> "CD4-positive, alpha-beta T cell"
  "CD8 T cells"
    -> "CD8-positive, alpha-beta T cell"
  "CD4-positive, CD25-positive, alpha-beta regulatory T cell"
    -> "regulatory T cell"
  "CD8-positive, alpha-beta memory T cell, CD45RO-positive"
    -> "CD8-positive, alpha-beta memory T cell"
  "mature alpha-beta T cell"
    -> "alpha-beta T cell"
  "mature NK T cell"
    -> "natural killer T cell"
  "mature B cell"
    -> "B cell"
  "effector memor

In [ ]:
# ── Apply LABEL_SYNONYM_MAP to all organs ─────────────────────────────────────
print('=' * 60)
print('  Applying LABEL_SYNONYM_MAP')
print('=' * 60)

total_remapped = 0
for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    col  = cfg['label_col']
    obs  = loaded_organs[name]['obs']
    organ_remapped = 0
    for src, tgt in LABEL_SYNONYM_MAP.items():
        n = (obs[col] == src).sum()
        if n > 0:
            loaded_organs[name]['obs'][col] = obs[col].replace(src, tgt)
            obs = loaded_organs[name]['obs']  # refresh ref after replace
            print(f'  [{name}] "{src}" -> "{tgt}"  ({n} cells)')
            organ_remapped += n
            total_remapped += n
    if organ_remapped == 0:
        print(f'  [{name}] no synonyms found')

print(f'\n  Total cells remapped : {total_remapped:,}')

# ── Re-check: any label in the synonym map that is still a KEY (not a value) ──
# If a canonical target also appears as a source, we have a chain — warn about it.
tgts = set(LABEL_SYNONYM_MAP.values())
chains = [k for k in LABEL_SYNONYM_MAP if k in tgts]
if chains:
    print(f'\n  WARNING: chained synonyms detected — these keys are also targets:')
    for c in chains:
        print(f'    {c}')
else:
    print('  [OK] No chained synonyms.')

# ── Post-unification label counts ─────────────────────────────────────────────
print('\n  Post-unification type counts per organ:')
for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    col  = cfg['label_col']
    n_types = loaded_organs[name]['obs'][col].nunique()
    n_cells = len(loaded_organs[name]['obs'])
    print(f'  {name:<20} {n_cells:>7,} cells  {n_types:>4} types')

print('[OK] Synonym unification done')

  Applying LABEL_SYNONYM_MAP
  [lung] "B cells" -> "B cell"  (300 cells)
  [lung] "NK cells" -> "natural killer cell"  (300 cells)
  [lung] "Alveolar macrophages" -> "alveolar macrophage"  (300 cells)
  [lung] "Mast cells" -> "mast cell"  (300 cells)
  [lung] "Plasma cells" -> "plasma cell"  (300 cells)
  [lung] "Classical monocytes" -> "classical monocyte"  (300 cells)
  [lung] "Non-classical monocytes" -> "non-classical monocyte"  (300 cells)
  [lung] "Plasmacytoid DCs" -> "plasmacytoid dendritic cell"  (300 cells)
  [lung] "Smooth muscle" -> "smooth muscle cell"  (300 cells)
  [lung] "CD4 T cells" -> "CD4-positive, alpha-beta T cell"  (300 cells)
  [lung] "CD8 T cells" -> "CD8-positive, alpha-beta T cell"  (300 cells)
  [brain_glia] no synonyms found
  [brain_neurons] no synonyms found
  [liver] "CD4-positive helper T cell" -> "CD4-positive, alpha-beta T cell"  (300 cells)
  [liver] "CD8-positive, alpha-beta memory T cell, CD45RO-positive" -> "CD8-positive, alpha-beta memory T cell"

: 

## 6. Build Combined Hierarchy

In [ ]:
print('=' * 60)
print('  Building combined cross-organ hierarchy')
print('=' * 60)

# ── A. Lung hierarchy (from ann_level_* columns) ──────────────────────────────
lung_cfg  = loaded_organs['lung']['cfg']
lung_obs2 = loaded_organs['lung']['obs']
lung_ontology = {}
level_cols = [c for c in lung_cfg['hierarchy_cols'] if c in lung_obs2.columns]
cols_ordered = level_cols + [lung_cfg['label_col']]
for k in range(1, len(cols_ordered)):
    parent_col, child_col = cols_ordered[k-1], cols_ordered[k]
    for _, row in lung_obs2[[parent_col, child_col]].dropna().drop_duplicates().iterrows():
        p, ch = str(row[parent_col]), str(row[child_col])
        if is_valid(p) and is_valid(ch) and ch not in lung_ontology:
            lung_ontology[ch] = p
for val in lung_obs2[cols_ordered[0]].dropna().unique():
    if is_valid(val) and str(val) not in lung_ontology:
        lung_ontology[str(val)] = None
print(f'  [lung] {len(lung_ontology)} ontology entries')

# ── B. Brain glia hierarchy (cell_type -> supercluster_term) ─────────────────
brain_glia_cfg = loaded_organs['brain_glia']['cfg']
brain_glia_obs = loaded_organs['brain_glia']['obs']
brain_glia_ontology = {}
coarse_col = brain_glia_cfg['coarse_col']
for _, row in brain_glia_obs[[coarse_col, brain_glia_cfg['label_col']]].dropna().drop_duplicates().iterrows():
    brain_glia_ontology[str(row[brain_glia_cfg['label_col']])] = str(row[coarse_col])
for val in brain_glia_obs[coarse_col].dropna().unique():
    if str(val) not in brain_glia_ontology:
        brain_glia_ontology[str(val)] = None
print(f'  [brain_glia] {len(brain_glia_ontology)} ontology entries')

# ── C. Cross-organ shared hierarchy ──────────────────────────────────────────
CROSS_ORGAN_HIERARCHY = {
    # ── Neuron subtypes (brain_neurons dataset) ──────────────────────────────
    'Upper-layer intratelencephalic':    'excitatory neuron',
    'Deep-layer intratelencephalic':     'excitatory neuron',
    'Deep-layer corticothalamic and 6b': 'excitatory neuron',
    'Deep-layer near-projecting':        'excitatory neuron',
    'Hippocampal CA1-3':                 'excitatory neuron',
    'Hippocampal CA4':                   'excitatory neuron',
    'Hippocampal dentate gyrus':         'excitatory neuron',
    'Thalamic excitatory':               'excitatory neuron',
    'Amygdala excitatory':               'excitatory neuron',
    'Upper rhombic lip':                 'excitatory neuron',
    'Lower rhombic lip':                 'excitatory neuron',
    'Mammillary body':                   'excitatory neuron',
    'CGE interneuron':                   'inhibitory neuron',
    'MGE interneuron':                   'inhibitory neuron',
    'Cerebellar inhibitory':             'inhibitory neuron',
    'LAMP5-LHX6 and Chandelier':         'inhibitory neuron',
    'Medium spiny neuron':               'inhibitory neuron',
    'Eccentric medium spiny neuron':     'inhibitory neuron',
    'Midbrain-derived inhibitory':       'inhibitory neuron',
    'Miscellaneous':                     'neuron',
    'excitatory neuron':                 'neuron',
    'inhibitory neuron':                 'neuron',
    # ── T cell lineage ───────────────────────────────────────────────────────
    'CD4-positive, alpha-beta T cell':                         'alpha-beta T cell',
    'CD8-positive, alpha-beta T cell':                         'alpha-beta T cell',
    'regulatory T cell':                                       'CD4-positive, alpha-beta T cell',
    'T follicular helper cell':                                'CD4-positive, alpha-beta T cell',
    'naive thymus-derived CD4-positive, alpha-beta T cell':    'CD4-positive, alpha-beta T cell',
    'central memory CD4-positive, alpha-beta T cell':          'CD4-positive, alpha-beta T cell',
    'effector memory CD4-positive, alpha-beta T cell':         'CD4-positive, alpha-beta T cell',
    'naive thymus-derived CD8-positive, alpha-beta T cell':    'CD8-positive, alpha-beta T cell',
    'central memory CD8-positive, alpha-beta T cell':          'CD8-positive, alpha-beta T cell',
    'effector memory CD8-positive, alpha-beta T cell':         'CD8-positive, alpha-beta T cell',
    'effector CD8-positive, alpha-beta T cell':                'CD8-positive, alpha-beta T cell',
    'gamma-delta T cell':                                      'T cell',
    'mucosal invariant T cell':                                'T cell',
    'natural killer T cell':                                   'T cell',
    'alpha-beta T cell':                                       'T cell',
    'T cell':                                                  'lymphocyte',
    # ── B cell lineage ───────────────────────────────────────────────────────
    'naive B cell':                   'B cell',
    'memory B cell':                  'B cell',
    'germinal center B cell':         'B cell',
    'plasmablast':                    'B cell',
    'plasma cell':                    'B cell',
    'transitional stage B cell':      'B cell',
    'B cell':                         'lymphocyte',
    # ── NK cells ─────────────────────────────────────────────────────────────
    'natural killer cell':            'lymphocyte',
    'innate lymphoid cell':           'lymphocyte',
    'lymphocyte':                     'leukocyte',
    # ── Myeloid lineage ──────────────────────────────────────────────────────
    'classical monocyte':             'monocyte',
    'non-classical monocyte':         'monocyte',
    'intermediate monocyte':          'monocyte',
    'monocyte':                       'myeloid leukocyte',
    'Kupffer cell':                   'macrophage',
    'alveolar macrophage':            'macrophage',
    'macrophage':                     'myeloid leukocyte',
    'conventional dendritic cell':    'dendritic cell',
    'plasmacytoid dendritic cell':    'dendritic cell',
    'dendritic cell':                 'myeloid leukocyte',
    'mast cell':                      'myeloid leukocyte',
    'neutrophil':                     'myeloid leukocyte',
    'basophil':                       'myeloid leukocyte',
    'eosinophil':                     'myeloid leukocyte',
    'myeloid leukocyte':              'leukocyte',
    'leukocyte':                      'Immune',
    # ── Hematopoietic progenitors (bone marrow) ───────────────────────────────
    'hematopoietic stem cell':        'hematopoietic precursor cell',
    'common myeloid progenitor':      'hematopoietic precursor cell',
    'granulocyte monocyte progenitor':'hematopoietic precursor cell',
    'common lymphoid progenitor':     'hematopoietic precursor cell',
    'hematopoietic precursor cell':   'Immune',
    # ── Erythroid lineage ────────────────────────────────────────────────────
    'proerythroblast':                'erythroid lineage cell',
    'erythroblast':                   'erythroid lineage cell',
    'reticulocyte':                   'erythroid lineage cell',
    'erythrocyte':                    'erythroid lineage cell',
    'erythroid lineage cell':         'hematopoietic precursor cell',
    # ── Megakaryocyte lineage ────────────────────────────────────────────────
    'megakaryocyte-erythroid progenitor cell': 'hematopoietic precursor cell',
    'megakaryocyte':                  'hematopoietic precursor cell',
    'platelet':                       'megakaryocyte',
    # ── Endothelial (cross-organ) ─────────────────────────────────────────────
    'endothelial cell of artery':             'endothelial cell',
    'endothelial cell of vein':               'endothelial cell',
    'endothelial cell of hepatic sinusoid':   'endothelial cell',
    'blood vessel endothelial cell':          'endothelial cell',
    'lymphatic endothelial cell':             'endothelial cell',
    'high endothelial venule cell':           'endothelial cell',
    'capillary endothelial cell':             'endothelial cell',
    # ── Fibroblast / Stroma (cross-organ) ────────────────────────────────────
    'hepatic stellate cell':          'fibroblast',
    'portal fibroblast':              'fibroblast',
    'fibroblastic reticular cell':    'fibroblast',
    # ── Liver-specific epithelial ─────────────────────────────────────────────
    'hepatocyte':                     'hepatic cell',
    'cholangiocyte':                  'hepatic cell',
    'hepatic cell':                   'Epithelial',
    # ── Cross-tissue bridges ─────────────────────────────────────────────────
    'Oligodendrocyte':                'glial cell',
    'Astrocyte':                      'glial cell',
    'Microglia':                      'glial cell',
    'Committed oligodendrocyte precursor': 'glial cell',
    'Oligodendrocyte precursor':      'glial cell',
    'Bergmann glia':                  'glial cell',
    'Choroid plexus':                 'glial cell',
    'Ependymal':                      'glial cell',
    'Fibroblast':                     'Fibroblast lineage',
}

# ── D. Merge all ontologies ───────────────────────────────────────────────────
combined_ontology = {}
combined_ontology.update(lung_ontology)
combined_ontology.update(brain_glia_ontology)
combined_ontology.update(CROSS_ORGAN_HIERARCHY)
for organ_name in ['liver', 'lymph_node', 'bone_marrow']:
    d   = loaded_organs[organ_name]
    col = d['cfg']['label_col']
    for ct in d['obs'][col].unique():
        ct = str(ct)
        if ct not in combined_ontology:
            combined_ontology[ct] = None  # root node

# ── E. Fix pericyte parent ────────────────────────────────────────────────────
old_parent = combined_ontology.get('pericyte', 'not set')
combined_ontology['pericyte'] = 'Vascular'
print(f'  [Fix] pericyte parent: "{old_parent}" -> "Vascular"')

# ── F. Deduplicate collision labels in lung ───────────────────────────────────
lung_obs3 = loaded_organs['lung']['obs']
lung_obs3, combined_ontology, dedup_report = deduplicate_hierarchy(
    lung_obs3, combined_ontology, lung_cfg['label_col']
)
loaded_organs['lung']['obs'] = lung_obs3
if dedup_report.empty:
    print('  [Fix] No collision labels — hierarchy already clean.')
else:
    print(f'  [Fix] Renamed {len(dedup_report)} collision label(s):')
    for _, row in dedup_report.iterrows():
        print(f'    "{row["original_label"]}" -> "{row["new_label"]}"  ({row["n_cells_renamed"]} cells)')

# ── G. Ensure synonym canonical targets are wired into ontology ───────────────
# After synonym unification, some canonical labels (e.g. 'CD4-positive, alpha-beta T cell')
# may now receive cells that previously had different names. The ontology already
# contains these canonical labels, so no extra wiring is needed — but we verify.
missing_in_ontology = []
for tgt in set(LABEL_SYNONYM_MAP.values()):
    if tgt not in combined_ontology:
        missing_in_ontology.append(tgt)
        combined_ontology[tgt] = None  # add as root if missing
if missing_in_ontology:
    print(f'  [Fix] Added {len(missing_in_ontology)} synonym targets to ontology as roots:')
    for m in missing_in_ontology:
        print(f'    {m}')
else:
    print('  [OK] All synonym targets already present in ontology.')

print(f'\n  Combined ontology entries : {len(combined_ontology)}')
pd.DataFrame(list(combined_ontology.items()), columns=['child', 'parent']).to_csv(
    os.path.join(OUT_DIR, 'ontology.csv'), index=False)
print('[OK] Combined hierarchy built')

## 7. Cell-to-Text Conversion

In [ ]:
print('=' * 60)
print('  Cell-to-text conversion (top-200 genes, no tissue prefix)')
print('=' * 60)
t0 = time.time()

all_texts      = []
all_labels_str = []
all_organ_ids  = []
organ_texts    = {}   # name -> list[str]
organ_labels   = {}   # name -> np.array

for cfg in ORGAN_CONFIGS:
    name      = cfg['name']
    d         = loaded_organs[name]
    label_col = cfg['label_col']
    n_cells   = len(d['valid_idx'])
    print(f'  [{name}] Converting {n_cells:,} cells ...', flush=True)
    t1 = time.time()

    texts = cell_to_text_backed(
        cfg['path'], d['valid_idx'], d['gene_symbols'], TOP_K_GENES, desc=name
    )

    # Filter empty texts
    keep_mask = [bool(t.strip()) for t in texts]
    texts     = [t for t, k in zip(texts, keep_mask) if k]
    labels    = d['obs'][label_col].astype(str).values[keep_mask]

    organ_texts[name]  = texts
    organ_labels[name] = labels

    all_texts.extend(texts)
    all_labels_str.extend(labels)
    all_organ_ids.extend([cfg['id']] * len(texts))

    print(f'  [{name}] {len(texts):,} cells kept | {time.time()-t1:.1f}s')

all_labels_str = np.array(all_labels_str)
all_organ_ids  = np.array(all_organ_ids, dtype=int)

print(f'\n  Total cells : {len(all_texts):,}')
print(f'  Total labels: {len(np.unique(all_labels_str))}')
print(f'  Elapsed     : {time.time()-t0:.1f}s')
print('[OK] Cell texts ready')

## 8. Class Vocabulary, Dataset, Splits

In [ ]:
print('=' * 60)
print('  Tokenizer, class vocabulary, dataset, splits')
print('=' * 60)
t0 = time.time()

print('  Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Build class vocabulary (leaves + all ancestor nodes in combined ontology)
leaf_classes_set = set(all_labels_str)
all_nodes = set()
for child, parent in combined_ontology.items():
    all_nodes.add(child)
    if parent: all_nodes.add(parent)
all_nodes |= leaf_classes_set

class_names   = sorted(all_nodes)
class_to_idx  = {name: idx for idx, name in enumerate(class_names)}
n_classes     = len(class_names)
leaf_classes  = sorted(leaf_classes_set)
leaf_indices  = [class_to_idx[c] for c in leaf_classes]
leaf_index_set = set(leaf_indices)

labels_encoded = np.array([class_to_idx[s] for s in all_labels_str], dtype=int)

print(f'  Total vocab (leaf + ancestors) : {n_classes}')
print(f'  Leaf classes                   : {len(leaf_classes)}')
for cfg in ORGAN_CONFIGS:
    n = len(set(organ_labels[cfg['name']]))
    print(f'    {cfg["name"]:<20} {n} leaf types')

class CellTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label':          torch.tensor(self.labels[idx], dtype=torch.long)}

# Stratified split
strat_key = labels_encoded * 10 + all_organ_ids
idx_all   = np.arange(len(all_texts))
idx_tv, idx_test   = train_test_split(idx_all, test_size=TEST_FRAC,
                                       stratify=strat_key, random_state=SEED)
idx_train, idx_val = train_test_split(idx_tv,  test_size=VAL_FRAC/(1-TEST_FRAC),
                                       stratify=strat_key[idx_tv], random_state=SEED)

train_labels = labels_encoded[idx_train]
val_labels   = labels_encoded[idx_val]
test_labels  = labels_encoded[idx_test]
test_organ   = all_organ_ids[idx_test]

train_ds = CellTextDataset([all_texts[i] for i in idx_train], train_labels, tokenizer, MAX_SEQ_LEN)
val_ds   = CellTextDataset([all_texts[i] for i in idx_val],   val_labels,   tokenizer, MAX_SEQ_LEN)
test_ds  = CellTextDataset([all_texts[i] for i in idx_test],  test_labels,  tokenizer, MAX_SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'\n  Train : {len(idx_train):,} | Val : {len(idx_val):,} | Test : {len(idx_test):,}')
print(f'  Train batches : {len(train_loader)}')
print(f'  Elapsed       : {time.time()-t0:.1f}s')
print('[OK] Datasets and DataLoaders ready')

## 9. HCE Reachability Matrix + Loss + Class Weights

In [ ]:
print('=' * 60)
print('  HCE reachability matrix + loss + class weights')
print('=' * 60)
t0 = time.time()

R_np = build_reachability_matrix_from_ontology(combined_ontology, class_names)
reachability_matrix = torch.tensor(R_np, dtype=torch.float32).to(device)

diag_ok = torch.allclose(torch.diag(reachability_matrix), torch.ones(n_classes, device=device))
nnz     = int(reachability_matrix.sum().item())
print(f'  Matrix shape : {n_classes} x {n_classes}')
print(f'  Diagonal OK  : {diag_ok}')
print(f'  Non-zeros    : {nnz}  ({nnz/(n_classes**2)*100:.1f}% density)')

# Class weights: w_i = N / (C * n_i) — inverse frequency
train_counts   = Counter(train_labels.tolist())
N_train, C_obs = len(train_labels), len(train_counts)
class_weights  = torch.zeros(n_classes, dtype=torch.float32, device=device)
for idx_ct, count in train_counts.items():
    class_weights[idx_ct] = N_train / (C_obs * count)

ancestor_indices = [i for i in range(n_classes) if i not in leaf_index_set]
for anc_idx in ancestor_indices:
    eff = sum(train_counts.get(j, 0) for j in leaf_indices if R_np[anc_idx, j] > 0)
    if eff > 0:
        class_weights[anc_idx] = N_train / (C_obs * eff)
class_weights = torch.clamp(class_weights, max=MAX_WEIGHT)

nz_w = (class_weights > 0).sum().item()
print(f'  Class weights: {nz_w}/{n_classes} non-zero | '
      f'range [{class_weights[class_weights>0].min():.4f}, {class_weights[class_weights>0].max():.4f}]')

class HCELoss(nn.Module):
    def __init__(self, R, class_weights=None, eps=1e-8):
        super().__init__()
        self.register_buffer('R', R)
        self.eps = eps
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None
    def forward(self, logits, targets):
        probs  = torch.softmax(logits, dim=1)
        s      = torch.clamp(probs @ self.R.T, min=self.eps)
        log_st = torch.log(s)[torch.arange(len(targets), device=targets.device), targets]
        if self.class_weights is not None:
            return -(self.class_weights[targets] * log_st).mean()
        return -log_st.mean()

criterion = HCELoss(reachability_matrix, class_weights).to(device)
print(f'  Elapsed : {time.time()-t0:.1f}s')
print('[OK] HCE loss ready')

## 10. Model

In [ ]:
print('=' * 60)
print('  Building model')
print('=' * 60)
t0 = time.time()

c2s_model = AutoModel.from_pretrained(C2S_MODEL_NAME)
c2s_model.gradient_checkpointing_enable()
hidden_size = c2s_model.config.hidden_size

class C2SClassifier(nn.Module):
    def __init__(self, encoder, hidden_size, num_classes):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.head    = nn.Linear(hidden_size, num_classes)
    def forward(self, input_ids, attention_mask):
        out         = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        seq_len     = attention_mask.sum(dim=1) - 1
        last_token  = last_hidden[torch.arange(last_hidden.size(0), device=last_hidden.device), seq_len]
        return self.head(self.dropout(last_token))

model = C2SClassifier(c2s_model, hidden_size, n_classes).to(device)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'  Hidden size    : {hidden_size}')
print(f'  Total params   : {total_params:.1f}M')
print(f'  Output classes : {n_classes}  (leaves + ancestors)')
print(f'  Elapsed        : {time.time()-t0:.1f}s')
print('[OK] Model ready')

## 11. Training

In [ ]:
print('=' * 60)
print('  Training')
print('=' * 60)
t_start = time.time()

optimizer        = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps      = len(train_loader) * N_EPOCHS
warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_STEPS)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS)

leaf_indices_t = torch.tensor(leaf_indices, device=device)
history        = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc   = 0.0
global_step    = 0

print(f'  Epochs        : {N_EPOCHS}')
print(f'  Steps/epoch   : {len(train_loader)}')
print(f'  Total steps   : {total_steps}')
print(f'  Warmup steps  : {WARMUP_STEPS}')

for epoch in range(1, N_EPOCHS + 1):
    t_epoch = time.time()
    model.train()
    running_loss, n_batches = 0.0, 0

    pbar = tqdm(train_loader, desc=f'  Epoch {epoch}/{N_EPOCHS} [train]', leave=True)
    for batch in pbar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_b       = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if global_step < WARMUP_STEPS: warmup_scheduler.step()
        else: cosine_scheduler.step()
        global_step  += 1
        running_loss += loss.item()
        n_batches    += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')

    train_loss = running_loss / n_batches

    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'  Epoch {epoch}/{N_EPOCHS} [val]  ', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_b       = batch['label'].to(device)
            logits         = model(input_ids, attention_mask)
            val_loss_sum  += criterion(logits, labels_b).item() * len(labels_b)
            leaf_logits    = logits[:, leaf_indices_t]
            preds          = leaf_indices_t[leaf_logits.argmax(dim=1)]
            val_correct   += (preds == labels_b).sum().item()
            val_total     += len(labels_b)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct  / val_total
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc}, BEST_MODEL_PATH)

    print(f'  Epoch {epoch}/{N_EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} | '
          f'val_acc={val_acc:.4f} | {"** BEST **" if improved else ""} | {time.time()-t_epoch:.0f}s')

print(f'\n  Total training time : {(time.time()-t_start)/60:.1f} min')
print(f'  Best val accuracy   : {best_val_acc:.4f}')
print('[OK] Training done')

## 12. Evaluation — Per-Organ Test Sets

In [ ]:
print('=' * 60)
print('  Evaluation on held-out test set')
print('=' * 60)

ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'  Loaded best checkpoint (epoch {ckpt.get("epoch","?")}, val_acc={ckpt.get("val_acc",0):.4f})')

model.eval()
test_preds, test_trues = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='  Evaluating', leave=True):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_b       = batch['label'].to(device)
        logits         = model(input_ids, attention_mask)
        leaf_logits    = logits[:, leaf_indices_t]
        preds          = leaf_indices_t[leaf_logits.argmax(dim=1)]
        test_preds.extend(preds.cpu().numpy())
        test_trues.extend(labels_b.cpu().numpy())

test_preds = np.array(test_preds)
test_trues = np.array(test_trues)

def evaluate_organ(trues, preds, organ_name, organ_id=None):
    if organ_id is not None:
        mask  = test_organ == organ_id
        trues = trues[mask]
        preds = preds[mask]
    if len(trues) == 0:
        print(f'  [{organ_name}] No test cells found')
        return pd.DataFrame()
    unique = np.unique(trues)
    acc    = accuracy_score(trues, preds)
    p, r, f, _ = precision_recall_fscore_support(trues, preds, labels=unique, average='macro', zero_division=0)
    p_pc, r_pc, f_pc, sup = precision_recall_fscore_support(trues, preds, labels=unique, zero_division=0)
    per_class = pd.DataFrame({
        'cell_type': [class_names[i] for i in unique],
        'precision': p_pc, 'recall': r_pc, 'f1': f_pc, 'support': sup,
    }).sort_values('recall', ascending=False).reset_index(drop=True)
    zero_recall = per_class[per_class['recall'] == 0.0]
    print(f'\n  [{organ_name}]')
    print(f'    Samples    : {len(trues):,}')
    print(f'    Accuracy   : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'    Macro F1   : {f:.4f}')
    print(f'    Macro Rec  : {r:.4f}')
    print(f'    0% recall  : {list(zero_recall["cell_type"]) if len(zero_recall) else "none"}')
    return per_class

per_class_results = {}
for cfg in ORGAN_CONFIGS:
    pc = evaluate_organ(test_trues, test_preds, cfg['name'], cfg['id'])
    per_class_results[cfg['name']] = pc
    if not pc.empty:
        pc.to_csv(os.path.join(OUT_DIR, f'{cfg["name"]}_per_class_metrics.csv'), index=False)

combined_pc = evaluate_organ(test_trues, test_preds, 'COMBINED')
combined_pc.to_csv(os.path.join(OUT_DIR, 'combined_per_class_metrics.csv'), index=False)
print('\n[OK] Evaluation complete')

## 13. Training Curves + Per-Organ Recall Plots

In [ ]:
epochs_ax = np.arange(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs_ax, history['train_loss'], 'o-', label='Train loss')
axes[0].plot(epochs_ax, history['val_loss'],   's-', label='Val loss')
axes[0].set(xlabel='Epoch', ylabel='HCE loss', title='Training Curves — Pan-Tissue v4.5')
axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(epochs_ax, history['val_acc'], 'o-', color='green', label='Val acc')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Validation Accuracy', ylim=[0, 1])
axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

def plot_recall(pc_df, title, fname):
    if pc_df.empty: return
    pc_sorted = pc_df.sort_values('recall')
    fig, ax   = plt.subplots(figsize=(10, max(4, len(pc_sorted) * 0.28)))
    colors    = ['#d73027' if r < 0.5 else '#fee090' if r < 0.8 else '#1a9850'
                 for r in pc_sorted['recall']]
    ax.barh(pc_sorted['cell_type'], pc_sorted['recall'], color=colors)
    ax.axvline(0.5, color='red',    ls='--', lw=1.2, label='50% recall')
    ax.axvline(0.8, color='orange', ls='--', lw=1.2, label='80% recall')
    ax.set(xlabel='Recall', title=f'Per-class Recall — {title}', xlim=[0, 1])
    ax.legend(fontsize=9); ax.tick_params(axis='y', labelsize=8); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150, bbox_inches='tight')
    plt.show()

for cfg in ORGAN_CONFIGS:
    plot_recall(
        per_class_results[cfg['name']],
        cfg['name'].replace('_', ' ').title(),
        f'{cfg["name"]}_recall.png'
    )

## 14. Zero-Shot Validation — All Lab Datasets

In [ ]:
print('=' * 60)
print('  Zero-shot inference on lab validation datasets')
print('=' * 60)

class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts, self.tokenizer, self.max_length = texts, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx] if self.texts[idx].strip() else '[PAD]',
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}

lab_results = {}

for lab_cfg in LAB_CONFIGS:
    lab_name  = lab_cfg['name']
    label_col = lab_cfg['label_col']
    print(f'\n  --- {lab_name} ---')
    t0 = time.time()

    adata_lab    = sc.read_h5ad(lab_cfg['path'])
    lab_gene_sym = get_gene_symbols(adata_lab)
    true_labels  = adata_lab.obs[label_col].astype(str).values

    X_lab = adata_lab.X
    texts = [
        cell_to_text_dense(X_lab[i], lab_gene_sym, TOP_K_GENES)
        for i in tqdm(range(adata_lab.n_obs), desc='    text', leave=False)
    ]

    inf_loader = DataLoader(
        InferenceDataset(texts, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True
    )

    preds, confs = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(inf_loader, desc='    infer', leave=False):
            logits      = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            leaf_logits = logits[:, leaf_indices_t]
            probs       = torch.softmax(leaf_logits, dim=1)
            pred_pos    = leaf_logits.argmax(dim=1)
            preds.extend(leaf_indices_t[pred_pos].cpu().numpy())
            confs.extend(probs.max(dim=1).values.cpu().numpy())

    pred_names = [class_names[i] for i in preds]
    confs      = np.array(confs)

    df = pd.DataFrame({
        'true_label': true_labels,
        'pred_label': pred_names,
        'confidence': confs,
    })
    safe_name = lab_name.replace(' ', '_').replace('(', '').replace(')', '')
    df.to_csv(os.path.join(OUT_DIR, f'lab_{safe_name}_predictions.csv'), index=False)
    lab_results[lab_name] = df

    breakdown = (
        df.groupby(['true_label', 'pred_label'])
        .size().reset_index(name='count')
    )
    breakdown['pct'] = breakdown.groupby('true_label')['count'].transform(lambda x: x / x.sum())
    breakdown.to_csv(os.path.join(OUT_DIR, f'lab_{safe_name}_breakdown.csv'), index=False)

    print(f'    Cells: {len(df):,} | Mean confidence: {confs.mean():.4f} | Elapsed: {time.time()-t0:.1f}s')
    unique_true = sorted(df['true_label'].unique())
    print(f'    Top predictions:')
    for true_lbl in unique_true:
        sub = breakdown[breakdown['true_label'] == true_lbl].sort_values('pct', ascending=False)
        n_total = sub['count'].sum()
        top = sub.iloc[0]
        print(f'      {true_lbl:<45} -> {top["pred_label"]:<40} {top["pct"]*100:.1f}%  (n={n_total})')

print('\n[OK] All lab dataset inference complete')

## 15. Lab Validation Visualizations

In [ ]:
def plot_lab_stacked_bar(df, lab_name):
    breakdown = (
        df.groupby(['true_label', 'pred_label'])
        .size().reset_index(name='count')
    )
    breakdown['pct'] = breakdown.groupby('true_label')['count'].transform(lambda x: x / x.sum())

    unique_true  = sorted(df['true_label'].unique())
    active_preds = df['pred_label'].value_counts().index.tolist()
    pivot = breakdown.pivot_table(
        index='true_label', columns='pred_label', values='pct', fill_value=0
    ).reindex(index=sorted(breakdown['true_label'].unique()), fill_value=0)

    palette   = plt.cm.get_cmap('tab20', len(active_preds))
    color_map = {c: palette(i) for i, c in enumerate(active_preds)}

    fig, ax = plt.subplots(figsize=(max(8, len(unique_true) * 0.9), 6))
    bottoms = np.zeros(len(pivot))
    x = np.arange(len(pivot))
    for pred_cls in active_preds:
        if pred_cls not in pivot.columns: continue
        vals = pivot[pred_cls].values
        ax.bar(x, vals, bottom=bottoms, label=pred_cls, color=color_map[pred_cls],
               edgecolor='white', linewidth=0.3)
        bottoms += vals
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Fraction of cells')
    ax.set_title(f'Model prediction mix — {lab_name}')
    ax.set_ylim(0, 1.05)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7,
              title='Predicted', title_fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    safe = lab_name.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(os.path.join(OUT_DIR, f'lab_{safe}_stacked_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()


def plot_lab_confidence(df, lab_name):
    unique_true = sorted(df['true_label'].unique())
    n_cols = min(4, len(unique_true))
    n_rows = int(np.ceil(len(unique_true) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 2.8))
    axes = np.array(axes).flatten()
    for i, true_lbl in enumerate(unique_true):
        conf = df[df['true_label'] == true_lbl]['confidence'].values
        axes[i].hist(conf, bins=20, range=(0, 1), color='steelblue', edgecolor='white', linewidth=0.4)
        axes[i].axvline(conf.mean(), color='red', ls='--', lw=1, label=f'mean={conf.mean():.2f}')
        axes[i].set_title(true_lbl, fontsize=8)
        axes[i].set_xlim(0, 1)
        axes[i].legend(fontsize=7)
        axes[i].tick_params(labelsize=7)
    for j in range(i + 1, len(axes)): axes[j].set_visible(False)
    fig.suptitle(f'Prediction confidence — {lab_name}', fontsize=11, y=1.01)
    plt.tight_layout()
    safe = lab_name.replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(os.path.join(OUT_DIR, f'lab_{safe}_confidence.png'), dpi=150, bbox_inches='tight')
    plt.show()


for lab_name, df in lab_results.items():
    print(f'\n  Plotting: {lab_name}')
    plot_lab_stacked_bar(df, lab_name)
    plot_lab_confidence(df, lab_name)

## 16. Summary

In [ ]:
print('=' * 60)
print('  PAN-TISSUE HCE v4.5 — TRAINING SUMMARY')
print('=' * 60)

print('\nTRAINING DATA')
print('=' * 40)
grand_total = 0
for cfg in ORGAN_CONFIGS:
    n = len(organ_texts[cfg['name']])
    t = len(set(organ_labels[cfg['name']]))
    print(f'  {cfg["name"]:<22} {n:>7,} cells  {t:>4} types')
    grand_total += n
print(f'  {"TOTAL":<22} {grand_total:>7,} cells')
print(f'  Leaf classes           : {len(leaf_classes)}')
print(f'  Total vocab (w/ anc.)  : {n_classes}')

print('\nSYNONYM UNIFICATION APPLIED')
print('=' * 40)
for src, tgt in LABEL_SYNONYM_MAP.items():
    print(f'  "{src}" -> "{tgt}"')

print('\nTEST SET PERFORMANCE')
print('=' * 40)
for cfg in ORGAN_CONFIGS:
    pc = per_class_results.get(cfg['name'], pd.DataFrame())
    if pc.empty: continue
    acc = accuracy_score(
        test_trues[test_organ == cfg['id']],
        test_preds[test_organ == cfg['id']]
    ) if (test_organ == cfg['id']).any() else 0
    min_r = pc['recall'].min()
    min_t = pc.loc[pc['recall'].idxmin(), 'cell_type']
    zeros = (pc['recall'] == 0).sum()
    print(f'  {cfg["name"]:<22} acc={acc:.3f}  min_recall={min_r:.3f} ({min_t})  zeros={zeros}')

print('\nZERO-SHOT LAB VALIDATION (dominant prediction)')
print('=' * 40)
for lab_name, df in lab_results.items():
    print(f'\n  {lab_name}')
    print(f'  Mean confidence: {df["confidence"].mean():.4f}')
    breakdown = (
        df.groupby(['true_label', 'pred_label'])
        .size().reset_index(name='count')
    )
    breakdown['pct'] = breakdown.groupby('true_label')['count'].transform(lambda x: x / x.sum())
    for true_lbl in sorted(df['true_label'].unique()):
        sub = breakdown[breakdown['true_label'] == true_lbl].sort_values('pct', ascending=False)
        top = sub.iloc[0]
        print(f'    {true_lbl:<45} -> {top["pred_label"]:<40} {top["pct"]*100:.1f}%')

print(f'\n[OK] Results saved to {OUT_DIR}/')